# 05 - 2026 Season Predictions

This notebook generates and tracks predictions for the 2026 NRL season using
the best trained model. It provides round-by-round predictions, season-to-date
accuracy tracking, a betting simulation, and current Elo standings.

**Sections:**
1. Load trained model
2. Load 2026 data and compute features
3. Generate predictions for next round
4. Display predictions table with probabilities
5. Season accuracy tracker
6. Betting simulation for 2026
7. Elo rating standings
8. Summary dashboard

In [ ]:
# ============================================================
# Cell 1: Imports and load trained model
# ============================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from datetime import datetime

from config.settings import (
    FEATURES_DIR, PROCESSED_DIR, OUTPUTS_DIR, PREDICT_YEAR,
    START_YEAR, END_YEAR, RANDOM_SEED,
)
from config.team_mappings import ALL_TEAMS
from modelling.model_registry import load_model, list_models, get_model_metadata

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------
# Load the best trained model from registry
# ------------------------------------------------------------------
print("Available models in registry:")
print("-" * 50)
saved_models = list_models()
if saved_models:
    for m in saved_models:
        print(f"  {m.get('name', '?')} v{m.get('version', '?')} "
              f"({m.get('model_type', '?')}) -- {m.get('saved_at', 'unknown date')}")
else:
    print("  No models found in registry.")

# Try to load the best model
model = None
model_name = None
feature_cols = None

# Look for a model with 'best' in its name first
for m in saved_models:
    if "best" in m.get("name", "").lower():
        model_name = m["name"]
        break

# Fall back to the most recently saved model
if model_name is None and saved_models:
    model_name = saved_models[-1]["name"]

if model_name:
    try:
        model = load_model(model_name)
        metadata = get_model_metadata(model_name)
        feature_cols = metadata.get("user_metadata", {}).get("feature_cols", None)
        print(f"\nLoaded model: {model_name}")
        print(f"Model type: {type(model).__name__}")
        if feature_cols:
            print(f"Feature columns: {len(feature_cols)} features")
    except Exception as e:
        print(f"Failed to load model '{model_name}': {e}")
else:
    print("\nNo model available. Please run notebook 03 first.")

print(f"\nPrediction year: {PREDICT_YEAR}")
print(f"Today: {datetime.now().strftime('%Y-%m-%d')}")

In [ ]:
# ============================================================
# Cell 2: Load current 2026 data and compute features
# ============================================================

from processing.feature_engineering import build_all_features

# Load all historical matches (including any 2026 data scraped so far)
matches_path = PROCESSED_DIR / "matches.parquet"
if matches_path.exists():
    all_matches = pd.read_parquet(matches_path)
else:
    alt = list(PROCESSED_DIR.glob("matches.*"))
    if alt:
        all_matches = pd.read_parquet(alt[0]) if alt[0].suffix == ".parquet" else pd.read_csv(alt[0])
    else:
        all_matches = pd.DataFrame()

# Load supplementary data
def load_if_exists(name):
    for ext in [".parquet", ".csv"]:
        p = PROCESSED_DIR / f"{name}{ext}"
        if p.exists():
            return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
    return None

ladders = load_if_exists("ladders")
lineups = load_if_exists("lineups")
odds_data = load_if_exists("odds")

year_col = "season" if "season" in all_matches.columns else "year"

if not all_matches.empty:
    print(f"Total matches loaded: {len(all_matches):,}")
    print(f"Seasons: {sorted(all_matches[year_col].dropna().unique())}")

    # Check how much 2026 data we have
    matches_2026 = all_matches[all_matches[year_col] == PREDICT_YEAR]
    print(f"\n2026 matches found: {len(matches_2026):,}")

    if not matches_2026.empty and "round" in matches_2026.columns:
        rounds_played = matches_2026["round"].dropna().unique()
        print(f"Rounds with data: {sorted(rounds_played)}")

        # Determine which matches have results vs upcoming
        if "home_score" in matches_2026.columns:
            completed = matches_2026[matches_2026["home_score"].notna()]
            upcoming = matches_2026[matches_2026["home_score"].isna()]
            print(f"Completed matches: {len(completed):,}")
            print(f"Upcoming matches (no score yet): {len(upcoming):,}")
        else:
            completed = matches_2026
            upcoming = pd.DataFrame()

    # Build features for all data (needed for rolling stats, Elo, etc.)
    print("\nComputing features for full dataset (including 2026)...")
    all_features = build_all_features(
        all_matches,
        ladders=ladders,
        lineups=lineups,
        odds=odds_data,
    )
    print(f"Features computed: {all_features.shape}")

    # Extract 2026 features
    features_2026 = all_features[all_features[year_col] == PREDICT_YEAR].copy()
    print(f"2026 feature rows: {len(features_2026):,}")
else:
    print("No match data available.")
    all_features = pd.DataFrame()
    features_2026 = pd.DataFrame()

In [ ]:
# ============================================================
# Cell 3: Generate predictions for next round
# ============================================================

if model is not None and not features_2026.empty:
    # Determine which feature columns to use
    META_COLS = {
        "home_team", "away_team", "venue", "date", "kickoff", "match_id",
        "season", "year", "round", "home_score", "away_score", "home_win",
        "target", "result", "y", "time_slot", "is_finals",
    }

    if feature_cols is None:
        feature_cols = [
            c for c in features_2026.select_dtypes(include=[np.number]).columns
            if c not in META_COLS and c != year_col
        ]

    # Find matches that need predictions (upcoming or all 2026 for tracking)
    # For upcoming matches, predict everything where we have features
    predict_df = features_2026.copy()

    # Check which features are available
    available_features = [c for c in feature_cols if c in predict_df.columns]
    missing_features = [c for c in feature_cols if c not in predict_df.columns]
    if missing_features:
        print(f"WARNING: {len(missing_features)} features missing from 2026 data:")
        print(f"  {missing_features[:10]}{'...' if len(missing_features) > 10 else ''}")
        # Fill missing features with 0 (or median from training data)
        for col in missing_features:
            predict_df[col] = 0
        available_features = feature_cols  # Use all expected features

    # Prepare feature matrix
    X_predict = predict_df[available_features].copy()

    # Fill NaN values with column median
    for col in available_features:
        if X_predict[col].isnull().any():
            median_val = X_predict[col].median()
            X_predict[col] = X_predict[col].fillna(median_val if pd.notna(median_val) else 0)

    # Generate predictions
    y_pred = model.predict(X_predict)
    y_prob = model.predict_proba(X_predict)

    # Handle predict_proba output shape
    if y_prob.ndim == 2:
        home_prob = y_prob[:, 1]
    else:
        home_prob = y_prob

    # Attach predictions to the DataFrame
    predict_df["pred_home_win"] = y_pred
    predict_df["prob_home_win"] = home_prob
    predict_df["prob_away_win"] = 1 - home_prob
    predict_df["predicted_winner"] = np.where(
        home_prob >= 0.5,
        predict_df["home_team"],
        predict_df["away_team"],
    )
    predict_df["confidence"] = np.maximum(home_prob, 1 - home_prob)

    # Find the latest round that needs predictions
    if "round" in predict_df.columns:
        latest_round = predict_df["round"].max()
        # If some rounds have results, find the first round without full results
        if "home_score" in predict_df.columns:
            rounds_with_results = (
                predict_df[predict_df["home_score"].notna()]["round"].unique()
            )
            all_rounds = sorted(predict_df["round"].dropna().unique())
            next_round = None
            for r in all_rounds:
                if r not in rounds_with_results:
                    next_round = r
                    break
            if next_round is None:
                next_round = latest_round
        else:
            next_round = latest_round
        print(f"Next round to predict: Round {next_round}")
    else:
        next_round = None

    print(f"\nGenerated predictions for {len(predict_df):,} matches")
    print(f"Home win predictions: {(y_pred == 1).sum()}/{len(y_pred)} ({(y_pred == 1).mean():.1%})")
    print(f"Average confidence: {predict_df['confidence'].mean():.1%}")
else:
    print("Cannot generate predictions: model or 2026 data not available.")
    predict_df = pd.DataFrame()

In [ ]:
# ============================================================
# Cell 4: Display predictions table with probabilities
# ============================================================

if not predict_df.empty and "round" in predict_df.columns:
    # Show predictions for the next round (or most recent round without results)
    if next_round is not None:
        round_preds = predict_df[predict_df["round"] == next_round].copy()
    else:
        round_preds = predict_df.tail(8).copy()  # Show last 8 matches

    if not round_preds.empty:
        display_df = round_preds[
            [c for c in ["round", "home_team", "away_team", "prob_home_win",
                         "prob_away_win", "predicted_winner", "confidence"]
             if c in round_preds.columns]
        ].copy()

        # Format probabilities as percentages
        if "prob_home_win" in display_df.columns:
            display_df["Home %"] = (display_df["prob_home_win"] * 100).round(1)
            display_df["Away %"] = (display_df["prob_away_win"] * 100).round(1)
            display_df["Conf %"] = (display_df["confidence"] * 100).round(1)

        print(f"\n{'='*70}")
        print(f"PREDICTIONS FOR ROUND {next_round} -- {PREDICT_YEAR} NRL SEASON")
        print(f"{'='*70}\n")

        show_cols = [c for c in ["home_team", "away_team", "Home %", "Away %",
                                  "predicted_winner", "Conf %"] if c in display_df.columns]

        display(
            display_df[show_cols]
            .reset_index(drop=True)
            .style
            .format({"Home %": "{:.1f}%", "Away %": "{:.1f}%", "Conf %": "{:.1f}%"})
            .background_gradient(subset=["Conf %"], cmap="RdYlGn", vmin=50, vmax=80)
        )

        # Visual representation
        fig, ax = plt.subplots(figsize=(12, max(4, len(round_preds) * 0.8)))

        matchups = []
        home_probs = []
        away_probs = []

        for _, row in round_preds.iterrows():
            ht = row.get("home_team", "Home")
            at = row.get("away_team", "Away")
            matchups.append(f"{ht}\nvs {at}")
            home_probs.append(row.get("prob_home_win", 0.5))
            away_probs.append(row.get("prob_away_win", 0.5))

        y_pos = np.arange(len(matchups))

        # Stacked horizontal bar chart
        bars_home = ax.barh(y_pos, home_probs, color="#1f77b4", label="Home Win",
                            edgecolor="white", height=0.6)
        bars_away = ax.barh(y_pos, [-ap for ap in away_probs], color="#ff7f0e",
                            label="Away Win", edgecolor="white", height=0.6)

        # Annotate
        for i, (hp, ap) in enumerate(zip(home_probs, away_probs)):
            ax.text(hp + 0.02, i, f"{hp:.0%}", va="center", fontsize=9, color="#1f77b4", fontweight="bold")
            ax.text(-ap - 0.02, i, f"{ap:.0%}", va="center", fontsize=9, color="#ff7f0e",
                    fontweight="bold", ha="right")

        ax.set_yticks(y_pos)
        ax.set_yticklabels(matchups, fontsize=9)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_xlim(-1, 1)
        ax.set_xlabel("Win Probability")
        ax.set_title(f"Round {next_round} Predictions - {PREDICT_YEAR}",
                     fontsize=14, fontweight="bold")
        ax.legend(loc="lower right")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{abs(x):.0%}"))

        plt.tight_layout()
        plt.show()
    else:
        print("No predictions available for the target round.")
else:
    print("Cannot display predictions: no round data available.")

In [ ]:
# ============================================================
# Cell 5: Season predictions tracker - accuracy so far
# ============================================================

if not predict_df.empty and "home_score" in predict_df.columns:
    # Filter to completed matches (those with results)
    completed = predict_df[predict_df["home_score"].notna()].copy()

    if not completed.empty:
        # Compute actual results
        completed["actual_home_win"] = (completed["home_score"] > completed["away_score"]).astype(int)
        completed["correct"] = (completed["pred_home_win"] == completed["actual_home_win"]).astype(int)

        overall_acc = completed["correct"].mean()
        total_correct = completed["correct"].sum()
        total_matches = len(completed)

        print(f"2026 SEASON ACCURACY TRACKER")
        print(f"{'='*50}")
        print(f"Matches completed: {total_matches:,}")
        print(f"Correct predictions: {total_correct:,}")
        print(f"Overall accuracy: {overall_acc:.1%}")

        # Accuracy by round
        if "round" in completed.columns:
            round_acc = (
                completed.groupby("round")
                .agg(
                    n=("correct", "count"),
                    correct=("correct", "sum"),
                    accuracy=("correct", "mean"),
                )
                .sort_index()
            )

            fig, axes = plt.subplots(1, 2, figsize=(16, 6))

            # Panel 1: Accuracy by round
            ax = axes[0]
            ax.plot(round_acc.index.astype(str), round_acc["accuracy"],
                    marker="o", markersize=8, linewidth=2, color="#1f77b4")

            # Cumulative accuracy
            cum_correct = completed.sort_values("round")["correct"].cumsum()
            cum_total = np.arange(1, len(completed) + 1)
            # Map back to rounds
            cum_acc_by_round = (
                completed.sort_values("round")
                .assign(cum_correct=lambda d: d["correct"].cumsum(),
                        cum_n=lambda d: np.arange(1, len(d) + 1),
                        cum_acc=lambda d: d["correct"].cumsum() / np.arange(1, len(d) + 1))
                .groupby("round")["cum_acc"].last()
            )
            ax.plot(cum_acc_by_round.index.astype(str), cum_acc_by_round.values,
                    linewidth=2, linestyle="--", color="#2ca02c", label="Cumulative")

            ax.axhline(0.5, color="red", linestyle=":", linewidth=0.8)
            ax.set_xlabel("Round")
            ax.set_ylabel("Accuracy")
            ax.set_title(f"2026 Prediction Accuracy by Round", fontsize=12, fontweight="bold")
            ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
            ax.legend()
            ax.set_ylim(0, 1)
            plt.xticks(rotation=45, ha="right")

            # Panel 2: Cumulative correct/incorrect
            ax = axes[1]
            ax.bar(round_acc.index.astype(str), round_acc["correct"],
                   label="Correct", color="#2ca02c", edgecolor="white")
            ax.bar(round_acc.index.astype(str),
                   round_acc["n"] - round_acc["correct"],
                   bottom=round_acc["correct"],
                   label="Incorrect", color="#d62728", edgecolor="white")

            for i, (rnd, row) in enumerate(round_acc.iterrows()):
                ax.text(i, row["n"] + 0.1, f"{row['accuracy']:.0%}",
                        ha="center", fontsize=8, fontweight="bold")

            ax.set_xlabel("Round")
            ax.set_ylabel("Matches")
            ax.set_title("Correct vs Incorrect by Round", fontsize=12, fontweight="bold")
            ax.legend()
            plt.xticks(rotation=45, ha="right")

            plt.tight_layout()
            plt.show()

            print("\nPer-round breakdown:")
            print("-" * 40)
            for rnd, row in round_acc.iterrows():
                print(f"  Round {str(rnd):>3s}: {row['correct']:.0f}/{row['n']:.0f} "
                      f"({row['accuracy']:.0%})")
    else:
        print("No completed 2026 matches with results yet.")
        print("Re-run this notebook after games are played to track accuracy.")
else:
    print("No 2026 prediction data with results available yet.")

In [ ]:
# ============================================================
# Cell 6: Betting simulation for 2026 season to date
# ============================================================

from evaluation.betting_simulation import BettingSimulator, plot_cumulative_pnl

if (not predict_df.empty
    and "home_score" in predict_df.columns
    and any(c in predict_df.columns for c in ["home_odds", "home_odds_close"])):

    completed = predict_df[predict_df["home_score"].notna()].copy()
    completed["actual_home_win"] = (completed["home_score"] > completed["away_score"]).astype(int)

    # Get odds column
    odds_col = "home_odds" if "home_odds" in completed.columns else "home_odds_close"
    away_odds_col = "away_odds" if "away_odds" in completed.columns else "away_odds_close"

    if odds_col in completed.columns and completed[odds_col].notna().any():
        sim_data = completed.dropna(subset=[odds_col]).copy()

        # Compute implied probabilities
        sim_data["home_implied"] = 1.0 / sim_data[odds_col]
        if away_odds_col in sim_data.columns:
            sim_data["away_implied"] = 1.0 / sim_data[away_odds_col]

        # Run flat-stake value betting simulation
        sim = BettingSimulator(initial_bankroll=1000, unit_stake=10)

        # Bet on home win when model probability > implied probability + edge
        home_bets = sim.flat_stake_value_bet(
            y_prob_model=sim_data["prob_home_win"].values,
            odds_implied_prob=sim_data["home_implied"].values,
            odds_decimal=sim_data[odds_col].values,
            y_true=sim_data["actual_home_win"].values,
            threshold=0.05,
        )

        summary = sim.get_summary()

        print("2026 BETTING SIMULATION (Flat-Stake Value Betting)")
        print("=" * 50)
        for key, val in summary.items():
            if key != "bets" and not isinstance(val, list):
                print(f"  {key:20s}: {val}")

        # Plot cumulative P&L
        if summary["total_bets"] > 0:
            results_dict = {"bets": sim.bets}
            fig = plot_cumulative_pnl(
                results_dict,
                title=f"2026 Season Cumulative P&L (Flat-Stake Value Betting)",
            )
            plt.show()
        else:
            print("\nNo value bets placed (model edge below threshold).")
    else:
        print("Odds data not available for betting simulation.")
else:
    print("Insufficient data for betting simulation.")
    print("Need: completed 2026 matches with odds data.")

In [ ]:
# ============================================================
# Cell 7: Elo rating standings for 2026
# ============================================================

# Check if Elo columns exist in features
elo_col_home = None
elo_col_away = None
for c in (all_features.columns if not all_features.empty else []):
    cl = c.lower()
    if "home" in cl and "elo" in cl and "diff" not in cl:
        elo_col_home = c
    if "away" in cl and "elo" in cl and "diff" not in cl:
        elo_col_away = c

if elo_col_home and elo_col_away and not all_features.empty:
    # Extract latest Elo rating for each team
    elo_records = []

    # Use all data up to the latest available point
    latest_data = all_features.tail(200)  # Last ~200 matches should cover recent form

    for _, row in latest_data.iterrows():
        elo_records.append({"team": row.get("home_team"), "elo": row.get(elo_col_home)})
        elo_records.append({"team": row.get("away_team"), "elo": row.get(elo_col_away)})

    elo_df = pd.DataFrame(elo_records).dropna(subset=["team", "elo"])

    # Take the latest Elo for each team
    latest_elo = elo_df.groupby("team")["elo"].last().sort_values(ascending=False)

    print(f"CURRENT ELO RATINGS ({PREDICT_YEAR} Season)")
    print("=" * 50)

    fig, ax = plt.subplots(figsize=(10, 8))

    # Color gradient based on Elo
    norm_elo = (latest_elo - latest_elo.min()) / (latest_elo.max() - latest_elo.min())
    colors = plt.cm.RdYlGn(norm_elo.values)

    y_pos = np.arange(len(latest_elo))
    ax.barh(y_pos, latest_elo.values, color=colors, edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(latest_elo.index, fontsize=9)
    ax.axvline(1500, color="gray", linestyle="--", linewidth=0.8, label="Starting rating (1500)")

    # Annotate with Elo values
    for i, (team, elo) in enumerate(latest_elo.items()):
        ax.text(elo + 5, i, f"{elo:.0f}", va="center", fontsize=9, fontweight="bold")

    ax.set_xlabel("Elo Rating")
    ax.set_title(f"NRL Elo Ratings - {PREDICT_YEAR} Season", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right")
    ax.invert_yaxis()  # Highest rated team at top

    plt.tight_layout()
    plt.show()

    # Print table
    print("\nElo Rankings:")
    print("-" * 50)
    for i, (team, elo) in enumerate(latest_elo.items(), 1):
        delta = elo - 1500
        arrow = "+" if delta >= 0 else ""
        print(f"  {i:2d}. {team:35s} {elo:>6.0f}  ({arrow}{delta:.0f})")
else:
    print("Elo data not available.")
    print("Elo features will be computed when the feature pipeline runs.")

In [ ]:
# ============================================================
# Cell 8: Summary dashboard
# ============================================================

print("=" * 70)
print(f"NRL {PREDICT_YEAR} PREDICTION DASHBOARD")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

# Model info
print(f"\nMODEL INFO")
print("-" * 40)
if model is not None:
    print(f"  Model: {model_name}")
    print(f"  Type: {type(model).__name__}")
    if feature_cols:
        print(f"  Features: {len(feature_cols)}")
else:
    print("  No model loaded.")

# Season status
print(f"\nSEASON STATUS")
print("-" * 40)
if not predict_df.empty:
    n_matches = len(predict_df)
    if "home_score" in predict_df.columns:
        n_completed = predict_df["home_score"].notna().sum()
        n_upcoming = predict_df["home_score"].isna().sum()
    else:
        n_completed = 0
        n_upcoming = n_matches

    print(f"  Total 2026 matches: {n_matches:,}")
    print(f"  Completed: {n_completed:,}")
    print(f"  Upcoming: {n_upcoming:,}")

    if n_completed > 0 and "home_score" in predict_df.columns:
        completed_2026 = predict_df[predict_df["home_score"].notna()].copy()
        completed_2026["actual"] = (completed_2026["home_score"] > completed_2026["away_score"]).astype(int)
        completed_2026["correct"] = (completed_2026["pred_home_win"] == completed_2026["actual"]).astype(int)
        acc = completed_2026["correct"].mean()

        print(f"\nACCURACY SO FAR")
        print("-" * 40)
        print(f"  Season accuracy: {acc:.1%} ({completed_2026['correct'].sum()}/{n_completed})")
else:
    print("  No 2026 data available yet.")

# Quick predictions summary
if not predict_df.empty and next_round is not None:
    round_matches = predict_df[predict_df["round"] == next_round]
    print(f"\nNEXT ROUND PREDICTIONS (Round {next_round})")
    print("-" * 40)
    for _, row in round_matches.iterrows():
        ht = row.get("home_team", "?")
        at = row.get("away_team", "?")
        winner = row.get("predicted_winner", "?")
        conf = row.get("confidence", 0) * 100
        print(f"  {ht:>30s} vs {at:<30s}  -> {winner} ({conf:.0f}%)")

print(f"\n{'='*70}")
print("Tip: Re-run this notebook after each round to update predictions and track accuracy.")
print("Run the scraping pipeline first to get the latest match results and odds.")